# Progetto pratico — Scienza ed etica dei dati

## Analisi etica del dataset ACSIncome

**Corso**: Scienza ed etica dei dati — **Prof.**: Riccardo Rosati — **A.A.**: 2025/2026 — **Autore**: Riccardo `<cognome>`

## Sezione 0 — Setup ambiente

Questa sezione prepara l'ambiente di esecuzione su Google Colab: installa le librerie non preinstallate, importa tutti i moduli usati nel notebook, fissa i seed per la riproducibilità e stampa le versioni delle dipendenze principali.

In [ ]:
# Installazione librerie non preinstallate su Colab.
# pandas, numpy, scikit-learn, matplotlib, scipy sono già disponibili.
!pip install -q folktables fairlearn shap lime imbalanced-learn diffprivlib seaborn

In [ ]:
# Import standard e scientific stack
import random
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import scipy.stats as stats

# scikit-learn: preprocessing, modelli, valutazione
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report,
)

# Dataset e moduli etici
import folktables
from folktables import ACSDataSource, ACSIncome

import fairlearn
from fairlearn.metrics import MetricFrame
from fairlearn.reductions import ExponentiatedGradient, DemographicParity, EqualizedOdds
from fairlearn.postprocessing import ThresholdOptimizer

import shap
import lime
import lime.lime_tabular

import diffprivlib
from diffprivlib.mechanisms import Laplace

# Riproducibilità: seed globali
SEED = 42
np.random.seed(SEED)
random.seed(SEED)

# Sopprime i FutureWarning / DeprecationWarning non bloccanti
# (shap e diffprivlib ne generano diversi su scikit-learn recente)
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=DeprecationWarning)

# Estetica plot
plt.style.use("seaborn-v0_8")
plt.rcParams["figure.figsize"] = (9, 5)
plt.rcParams["figure.dpi"] = 100
sns.set_palette("colorblind")

In [ ]:
# Stampa versioni librerie principali per riproducibilità
import sys
import sklearn
import importlib.metadata

def get_version(module, name=None):
    """Restituisce la versione di una libreria.
    Prova prima module.__version__, poi importlib.metadata per
    le librerie che non espongono __version__ (es. lime).
    """
    try:
        return module.__version__
    except AttributeError:
        return importlib.metadata.version(name or module.__name__)

versions = {
    "python": sys.version.split()[0],
    "numpy": np.__version__,
    "pandas": pd.__version__,
    "scikit-learn": sklearn.__version__,
    "fairlearn": fairlearn.__version__,
    "shap": shap.__version__,
    "lime": get_version(lime, "lime"),
    "diffprivlib": diffprivlib.__version__,
    "folktables": get_version(folktables, "folktables"),
}

for name, ver in versions.items():
    print(f"{name:>14s}: {ver}")

### Sezione 1 — Introduzione e framing etico

Il dataset ACSIncome — derivato dall'American Community Survey del U.S. Census Bureau — nasce per uno scopo statistico-aggregato: descrivere la struttura economica della popolazione. Da quando Ding, Hardt, Miller e Schmidt (2021) lo hanno proposto come benchmark per la ricerca su fairness, è diventato un terreno su cui vengono addestrati modelli predittivi reimpiegabili, in linea di principio, per scoring creditizio o valutazione assicurativa. Questo passaggio — da dato pubblico ad alta granularità a modello operativo su decisioni individuali — è il caso paradigmatico della **dual-use dei dati**: l'oggetto è uno, le sue conseguenze etiche sono due.

Questo progetto si colloca esattamente in quel passaggio. Non costruiamo un sistema di produzione, ma compiamo l'audit che andrebbe compiuto **prima** che un sistema del genere venga messo in funzione. Affrontiamo due delle quattro aree etiche del corso: **bias ed equità** (parte A) e **protezione della privacy** (parte B); l'interpretabilità via SHAP e LIME è trattata come strumento interno all'audit di bias. L'area di *autenticità umano/artificiale* resta fuori dallo scope per coerenza con il tipo di dato.

Il framing adottato è quello dell'*etica interna al design*: non una verifica di conformità a posteriori, ma una serie di scelte tecniche che incorporano valori nel processo stesso. Metriche di fairness, mitigazione del bias, k-anonymity e differential privacy non sono *soluzioni* a un problema etico esterno: sono **strumenti che abilitano scelte etiche**. Quale definizione di fairness adottare, quale ε di privacy accettare, quale trade-off privilegiare — sono questioni che la tecnica permette di formulare con precisione, ma che richiedono un giudizio che la tecnica non può sostituire.

Il punto di arrivo è la verifica di un'ipotesi: quando si cerca contemporaneamente di mitigare il bias, proteggere la privacy e mantenere l'utility di un modello, si incontra un **trilemma** strutturale che la sezione 14 affronta esplicitamente.

## Sezione 2 — Dataset: descrizione e attributi

**ACSIncome** è un benchmark proposto da Ding, Hardt, Miller e Schmidt nel paper *Retiring Adult: New Datasets for Fair Machine Learning* (NeurIPS 2021), pensato come sostituto moderno del classico UCI Adult. È derivato dall'**American Community Survey** (ACS) del U.S. Census Bureau, un'indagine annuale sulla popolazione statunitense con un campione molto più ampio, recente e dettagliato del census decennale.

Il **task** è una classificazione binaria: dato il profilo socio-economico di un individuo, prevedere se il reddito personale (`PINCP`) supera i 50.000 USD all'anno.

Lavoriamo su **due stati**: **California** (alto reddito medio, alta diversità etnica, costo della vita elevato) e **Mississippi** (basso reddito medio, demografia molto diversa, contesto socio-economico opposto). Questo confronto è il cuore della narrativa **context-dependent fairness**: vogliamo verificare se il bias che misureremo nelle sezioni successive è una proprietà del modello/dato in sé o se dipende sensibilmente dal contesto in cui viene applicato.

Nella tabella alla fine di questa sezione classifichiamo ogni colonna in tre categorie: **target**, **attributi sensibili** (centro della Parte A — Bias) e **quasi-identificatori** (centro della Parte B — Privacy).

In [ ]:
# Configurazione della fonte dati ACS (American Community Survey 2018, 1-Year)
data_source = ACSDataSource(survey_year='2018', horizon='1-Year', survey='person')

# Scaricamento separato per CA e MS, con aggiunta manuale della colonna STATE
# (df_to_pandas scarta la colonna ST perché non è feature del task ACSIncome).
features_per_state = []
labels_per_state = []

for stato in ['CA', 'MS']:
    # Download dei dati grezzi dello stato
    raw = data_source.get_data(states=[stato], download=True)
    # Estrazione di (features, label, group) secondo la specifica ACSIncome
    feat_df, lab_df, _ = ACSIncome.df_to_pandas(raw)
    # Aggiunta colonna narrativa STATE (valore costante per ogni stato)
    feat_df = feat_df.copy()
    feat_df['STATE'] = stato
    features_per_state.append(feat_df)
    labels_per_state.append(lab_df)

# Concatenazione dei due stati in un unico dataset
features_full = pd.concat(features_per_state, ignore_index=True)
label_full = pd.concat(labels_per_state, ignore_index=True)

# Verifica di coerenza tra features e label
assert len(features_full) == len(label_full), "Mismatch features/label"

# Riepilogo del dataset completo (pre-subsample)
print(f"Shape totale features: {features_full.shape}")
print(f"Shape totale label:    {label_full.shape}")
print(f"\nDistribuzione per STATE:\n{features_full['STATE'].value_counts()}")
print(f"\nDistribuzione target (income > 50k):\n{label_full.iloc[:, 0].value_counts(normalize=True).round(3)}")

In [ ]:
# Subsample stratificato per STATE: 25.000 record per CA + 25.000 per MS
# (groupby+sample è più chiaro di train_test_split quando si vogliono numeri
#  esatti per gruppo; il risultato è bilanciato 25k+25k per costruzione).
N_PER_STATE = 25_000

sampled_idx = (
    features_full
    .groupby('STATE', group_keys=False)
    .apply(lambda g: g.sample(n=N_PER_STATE, random_state=SEED))
    .index
)

df_features = features_full.loc[sampled_idx].reset_index(drop=True)
df_label = label_full.loc[sampled_idx].reset_index(drop=True)

# Verifiche finali
assert len(df_features) == 50_000
assert len(df_label) == 50_000
assert (df_features['STATE'].value_counts() == N_PER_STATE).all()

print(f"Shape df_features: {df_features.shape}")
print(f"Shape df_label:    {df_label.shape}")
print(f"\nBilanciamento STATE (subsample):\n{df_features['STATE'].value_counts()}")
print(f"\nDistribuzione target nel subsample:\n{df_label.iloc[:, 0].value_counts(normalize=True).round(3)}")

In [ ]:
# Classificazione esplicita di ogni colonna del dataset secondo le 4 categorie
# usate nel progetto: target, sensibile (Parte A), quasi-ID (Parte B), normale.
attr_classification = [
    ('PINCP',  'target',               'Reddito personale binarizzato (> 50.000 USD)'),
    ('SEX',    'sensibile',            'Sesso (1 = maschio, 2 = femmina)'),
    ('RAC1P',  'sensibile',            'Race code (1-9, categoriale)'),
    ('AGEP',   'quasi-identificatore', 'Età in anni'),
    ('SCHL',   'quasi-identificatore', 'Livello di istruzione (categoriale ordinato)'),
    ('OCCP',   'quasi-identificatore', 'Codice occupazione (categoriale ad alta cardinalità)'),
    ('POBP',   'quasi-identificatore', 'Place of birth (luogo di nascita)'),
    ('MAR',    'quasi-identificatore', 'Stato civile (marital status)'),
    ('COW',    'feature normale',      'Class of worker (settore di impiego)'),
    ('RELP',   'feature normale',      'Relazione con persona di riferimento del nucleo'),
    ('WKHP',   'feature normale',      'Ore settimanali lavorate (usual hours worked)'),
    ('STATE',  'feature normale',      'Codice stato (CA / MS) — variabile narrativa'),
]
attr_table = pd.DataFrame(attr_classification, columns=['feature', 'tipo', 'descrizione'])
print(attr_table.to_string(index=False))

La distinzione tra **attributi sensibili** e **quasi-identificatori** è il cardine metodologico del progetto: i primi (`SEX`, `RAC1P`) sono il bersaglio dell'audit di fairness nella Parte A, perché definiscono i sottogruppi rispetto ai quali misuriamo disparate impact, equal opportunity ecc.; i secondi (`AGEP`, `SCHL`, `OCCP`, `POBP`, `MAR`) sono il bersaglio della Parte B, perché abilitano attacchi di re-identificazione anche su un dataset apparentemente anonimo. La colonna `STATE` non rientra in nessuna delle due categorie: non è un attributo sensibile in senso GDPR né un quasi-ID realistico, è una **variabile narrativa** che useremo solo per il confronto contestuale CA vs MS.